In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import copy
from tqdm import tqdm

In [2]:
# Dataset directory relative to this script
data_dir = './dataset'

# Device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
image_size = 224
batch_size = 32
num_epochs = 10
learning_rate = 1e-4

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load datasets
train_dataset = datasets.ImageFolder(f'{data_dir}/Train', transform=train_transform)
val_dataset = datasets.ImageFolder(f'{data_dir}/Validation', transform=test_transform)
test_dataset = datasets.ImageFolder(f'{data_dir}/Test', transform=test_transform)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

Using device: cuda


In [3]:
def train_model(model, train_loader, val_loader, epochs=num_epochs, lr=learning_rate):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_iter = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training")
        for imgs, labels in train_iter:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            train_iter.set_postfix(loss=loss.item())

        epoch_loss = running_loss / len(train_loader.dataset)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            val_iter = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Validation")
            for imgs, labels in val_iter:
                imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(imgs)
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f} Val Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, 'best_model.pth')
            print("Best model saved.")

    print(f"Training complete. Best Val Acc: {best_val_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model

def evaluate_model(model, test_loader):
    model = model.to(device)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        test_iter = tqdm(test_loader, desc="Testing")
        for imgs, labels in test_iter:
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    acc = correct / total
    print(f"Test Accuracy: {acc:.4f}")
    return acc

In [5]:
effnet_model = timm.create_model('efficientnet_b0', pretrained=True)
effnet_model.classifier = nn.Linear(effnet_model.classifier.in_features, 2)
print("Training EfficientNet...")
effnet_model = train_model(effnet_model, train_loader, val_loader)
torch.save(effnet_model.state_dict(), 'efficientnet_model_final.pth')
evaluate_model(effnet_model, test_loader)

Training EfficientNet...


Epoch 1/10 - Validation: 100%|██████████| 95/95 [00:18<00:00,  5.07it/s]


Epoch [1/10] Loss: 0.1891 Val Acc: 0.9517
Best model saved.


Epoch 2/10 - Validation: 100%|██████████| 95/95 [00:18<00:00,  5.25it/s]


Epoch [2/10] Loss: 0.0574 Val Acc: 0.9683
Best model saved.


...

Epoch 10/10 - Validation: 100%|██████████| 95/95 [00:18<00:00,  5.25it/s]


Epoch [10/10] Loss: 0.0078 Val Acc: 0.9676
Training complete. Best Val Acc: 0.9755


Testing: 100%|██████████| 32/32 [00:07<00:00,  4.46it/s]

Test Accuracy: 0.8343


0.8343253968253969

In [ ]:
#DeiT model
deit_model = timm.create_model('deit_small_patch16_224', pretrained=True)
deit_model.head = nn.Linear(deit_model.head.in_features, 2)
print("Training DeiT...")
deit_model = train_model(deit_model, train_loader, val_loader)
torch.save(deit_model.state_dict(), 'deit_model_final.pth')
evaluate_model(deit_model, test_loader)

Training DeiT...


Epoch 1/10 - Validation: 100%|██████████| 95/95 [00:23<00:00,  4.10it/s]


Epoch [1/10] Loss: 0.2058 Val Acc: 0.9550
Best model saved.


Epoch 2/10 - Validation: 100%|██████████| 95/95 [00:23<00:00,  4.06it/s]


Epoch [2/10] Loss: 0.0873 Val Acc: 0.9679
Best model saved.


...

Epoch 10/10 - Validation: 100%|██████████| 95/95 [00:23<00:00,  3.97it/s]


Epoch [10/10] Loss: 0.0232 Val Acc: 0.9666
Training complete. Best Val Acc: 0.9679


Testing: 100%|██████████| 32/32 [00:08<00:00,  3.95it/s]

Test Accuracy: 0.8641


0.8640873015873016

In [ ]:
from PIL import Image
import torch
from torchvision import transforms
import timm

#Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Specify image path
image_path = './test-images/test_vansh.jpg'

# Define the same preprocessing used during training
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load and preprocess the image
img = Image.open(image_path).convert('RGB')
img_tensor = test_transform(img).unsqueeze(0).to(device)

# Instantiate the model architecture (replace with your architecture)
model = timm.create_model('efficientnet_b0', pretrained=False)
model.classifier = torch.nn.Linear(model.classifier.in_features, 2)

# Load the trained weights
model.load_state_dict(torch.load('efficientnet_model_final.pth', map_location='cpu'))  # Adjust path and device if needed
model.eval()
model.to(device)

# Make prediction
with torch.no_grad():
    output = model(img_tensor)
    prob = torch.softmax(output, dim=1)
    pred_class = torch.argmax(prob, dim=1).item()

# Map class index to label
class_map = {0: 'Fake', 1: 'Real'}
print(f'Prediction: {class_map[pred_class]}, Probability: {prob[0, pred_class].item():.4f}')


Prediction: Real, Probability: 1.0000


C:\Users\sriva\AppData\Local\Temp\ipykernel_2840\2623675269.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('efficientnet_model_final.p